In [ ]:
%matplotlib inline

# Version 11 mai 2026
import os, fnmatch, sys
import io, contextlib
import glob
import numpy as np
import numpy.ma as ma

import pandas as pd
import geopandas as gp
from pathlib import Path
from jupyprint import jupyprint  
from IPython.display import display, FileLink, FileLinks, HTML
from yaml import safe_load
from copy import copy

# matplotlib
from matplotlib import colors
from pylab import *
from IPython.display import Image
# from IPython.display import display
# import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

from itertools import accumulate
from operator import add 

from tqdm.auto import tqdm  # au lieu de: from tqdm import tqdm
from yaml import safe_dump
from scipy.interpolate import RegularGridInterpolator
from shapely import LineString, intersection

################
# Clawpack     #
################
# Source : http://www.clawpack.org/gallery/_static/apps/notebooks/visclaw/gridtools.html#Extract-1d-transects
from clawpack.visclaw import colormaps, frametools, geoplot, gridtools
from clawpack.visclaw import animation_tools
from clawpack.visclaw.plottools import pcolorcells
from clawpack.geoclaw import dtopotools, topotools, marching_front
from clawpack.geoclaw import util
from clawpack.pyclaw.solution import Solution
from clawpack.pyclaw import solution as solution
from clawpack.amrclaw import region_tools
from clawpack.geoclaw import fgout_tools

################
# Latex        #
################
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['text.latex.preamble'] = r'\usepackage{libertine}'
plt.rcParams['font.size'] = 12

################
# Répertoires  #
################
current_directory = Path().cwd()
proj_dir      = current_directory.parent
topo_dir      = proj_dir / "Topo"
images_dir    = proj_dir / "Figures"
animation_dir = images_dir / 'Animation'
video_dir     = images_dir / 'Vidéos'
output_dir    = current_directory / "_output"
BC_dir        = proj_dir / "CL"
export_dir    = proj_dir / 'Export'
avac_dir      = proj_dir / "AVAC"
répertoires   = {'proj_dir':str(proj_dir),'topo_dir':str(topo_dir),
                 'avac_dir':str(avac_dir),'images_dir':str(images_dir),
                 'output_dir':str(output_dir),'waves_dir':str(current_directory)}
if not animation_dir.exists():
     animation_dir.mkdir(parents=True, exist_ok=True)
     print(f"Création du répertoire {animation_dir}")
if not video_dir.exists():
     video_dir.mkdir(parents=True, exist_ok=True)
     print(f"Création du répertoire {video_dir}")
HOME       = current_directory
répertoire_travail  = HOME 
répertoire_résultat = répertoire_travail / '_output'
configuration_file  = proj_dir / "impulse_configuration.yaml"
configuration_avac  = avac_dir / "AVAC_configuration.yaml"
sys.path.insert(0, str(proj_dir))

##################
# Configurations #
##################
if configuration_file.exists():
    print(f"Chargement du fichier de configuration existant")
    with open(configuration_file) as file:
        config        = safe_load(file)
        topo          = config["topo_files"]
        lake          = config["lake"]
        gauges        = config['gauges']
        rheology      = config['rheology']
        computation   = config['computation']
        output        = config['output']
        topo_files    = config['topo_files']
        scenario      = config['scenario']
        period_return = scenario['period_return']
        directory     = config['directory']
else:
    print(f"Le fichier {configuration_file} est manquant. Il faut le générer avec Waves_preprocess")
print('Répertoire où sont les données : \n  %s' % répertoire_résultat)
if not os.path.isdir(répertoire_travail):
      print('*** Revoir le nom du répertoire (je ne le trouve pas)')

if configuration_avac.exists():
    print(f"Chargement du fichier de configuration AVAC existant")
    with open(configuration_avac) as file:
        avac_config        = safe_load(file)
else:
    print(f"Le fichier {configuration_avac} est manquant. Il faut le générer avec le cahier AVAC")
config_total = {'avac':avac_config,'waves':config,'répertoires':répertoires}
################
# Module waves #
################
from module_waves import create_boundary_conditions, reload_wave, check_configuration, check_output, boundary_values
from module_waves import Create_Animation, plot_four_gauge_profiles
from module_waves import calculate_overflow_rate, create_plot_lake_contour_meshing, calculate_inflow_rate, calculate_inflow
from module_waves import compare_dicts
from module_waves import calculate_wave_energy

################
# AVAC         #
################
sys.path.insert(0, str(proj_dir / "AVAC"))
from module_avac import raster_read_features, avac_reload, raster_read_file, \
    claw_export_dem, claw_export_dem_window, raster_determine_type, raster_plot_topo, make_output
from module_avac import fn_eta, fn_ground, fn_h, fn_hu, fn_husquare, fn_hv, fn_u, fn_v, fn_extract
from module_avac import format_m
      
################
# Variables    #
################
CLAW           = os.environ['CLAW']
format_fichier = output['output_format']
rho            = rheology['rho']
g              = rheology['gravity']
z_lac          = lake['water_level'] if configuration_file.exists() else 1500
nb_grille      = computation['nb_grid'] # nombre d'éléments de la grille d'interpolation
# paramètre d'export des figures
figure_export_params = dict(dpi = 300, bbox_inches = 'tight')

# topo
topo_fine = raster_read_file(str(topo_dir / topo_files['fine']))
xmin_mnt, xmax_mnt, ymin_mnt, ymax_mnt, nbx_mnt, nby_mnt, cell_size_mnt, dictionary_mnt, _, _, _, _ = \
        raster_read_features(topo_dir / lake['topography'])
if check_configuration(config,topo_dir):
     print("Pas d'erreur détectée dans le fichier de configuration.")

# période de retour
avac_period = avac_config['release']['period_return']
wave_period = config['scenario']['period_return']
if avac_period != wave_period:
     print("Attention les périodes de retour ne coïncident pas")
print(f"Période de retour considérée :\n * T= {avac_period} ans pour AVAC \n * T= {wave_period} ans pour Wave")
print(f"Répertoire de résultats : \n* AVAC : {avac_config['output']['output_directory']}\n* Wave : {output['output_directory']}")


# fichier de temps (avalanche entrant dans le lac)
df_temps_dir  = export_dir / 'df_temps_avalanche.csv'

if df_temps_dir.exists():
    df_temps = pd.read_csv(df_temps_dir)
    print(f"Lecture du fichier {df_temps_dir}")
else:
    import warnings
    warnings.filterwarnings('ignore', category=FutureWarning, module='pandas')
    df_temps = pd.DataFrame(columns=['T', 'eps', 't_déb', 't_fin'])
    print(f"Création de df_temps (fichier des temps avalanche).")



# Topographie

In [ ]:
# Ensemble des données topo
topo_file = raster_read_file(topo_dir / lake['topography'])

fig, ax, x0, y0 = raster_plot_topo(topo_file,step=500,resampling=5)
ax.add_patch(plt.Rectangle((lake['xmin']-x0,lake['ymin']-y0) , width=lake['xmax']-lake['xmin'], 
                           height=lake['ymax']-lake['ymin'], ls="-", lw=2, ec="red", fc="none"))

In [ ]:
# vérification du masque de contour et des jauges
# Ensemble des données topo et zoom sur le lac
lake_mask_gdf  = gp.read_file(topo_dir / topo_files['mask_shp']) # importation du masque
crop_map       = True   # réduire la carte aux dimensions de la zone zoomée
step_plot_topo = config['output']['carto_layout']['minor_label_step']
text_m         = 10     # décalage du texte pour les étiquettes
margin         = config['output']['carto_layout']['margin']
crop_region    = (lake['xmin'] - margin, lake['xmax'] + margin,
                                 lake['ymin'] - margin, lake['ymax'] + margin)
if crop_map:
    topo_file_crop  = topo_file.crop(filter_region = crop_region)
else:
    topo_file_crop = copy(topo_file)
fig, ax, x0, y0 = raster_plot_topo(topo_file_crop, step = step_plot_topo)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=2, ec="red", fc="none"))

# Zoom sur le lac avec une marge
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)

# tracé du masque
lake_mask_gdf.geometry.translate(-x0, -y0).plot(
    ax=ax, facecolor='none', edgecolor='blue', linewidth=1.5 
)

if gauges['gauge_recording']:
    for k in range(len(gauges)-1):
        x = gauges[str(k)]['x']
        y = gauges[str(k)]['y']
        ax.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)
        ax.text(x-x0,y-y0+text_m,str(1+k),color = 'red')

fig.savefig(images_dir / "vue_domaine_calcul.png",**figure_export_params)





In [ ]:
# Extension du MNT complet
print(f"Domaine du MNT :")
print(f"- xmin = {format_m(xmin_mnt)} m et xmax = {format_m(xmax_mnt)}")
print(f"- ymin = {format_m(ymin_mnt)} m et ymax = {format_m(ymax_mnt)}")
print(f"Domaine du Lac :")
print(f"- xmin = {format_m(lake['xmin'])} m et xmax = {format_m(lake['xmax'])}, Δx = {format_m(lake['xmax']-lake['xmin'])} m")
print(f"- ymin = {format_m(lake['ymin'])} m et ymax = {format_m(lake['ymax'])}, Δy = {format_m(lake['ymax']-lake['ymin'])} m")


# Calcul

In [ ]:
# afficher la configuration pour une ultime vérification
afficher_configuration = False
if afficher_configuration:
     from module_waves import afficher_config
     afficher_config(config)
reload_wave()
from module_waves import compare_dicts

cmp = compare_dicts(config, proj_dir / 'impulse_configuration.yaml')

In [ ]:
# Running geoclaw
make_output(config)

In [ ]:
# vérification de la masse de fluide présente dans le domaine de calcul
plt.close('all')  # Ferme les figures widget avant de repasser en inline
# Plot with contour lines
%matplotlib inline
mass_evolution = pd.read_csv(output_dir/'mass.txt')

text_legend = {'French':r'Masse de fluide (en t)',
               'English':r'Fluid mass (t)'}[config['output']['language']]
 

fig, ax = plt.subplots(figsize=(8,4))
#ax.plot(mass_evolution['t_s'], mass_evolution['wet_mass_kg'],    label='masse dans le domaine')
ax.plot(mass_evolution['t_s'], mass_evolution['wet_mass_kg']/1e6 )
ax.set_xlabel('$t$ (s)')

ax.set_ylabel(rf'{text_legend}')
#ax.legend()
ax.grid()
fig.savefig(images_dir / "evolution_total_mass_lake_Wave.png",**figure_export_params)

# Importation des résultats

In [ ]:
# vérification des sorties
fichiers_fort = fnmatch.filter(os.listdir(répertoire_résultat), 'fort.q*' )
fichiers      = fnmatch.filter(os.listdir(répertoire_résultat), 'fgout*.q*' )
NbSim    = len(fichiers)-1
if NbSim != computation['nb_simul']:
    print(f"Il y avait {computation['nb_simul']} simulations prévues, et j'en trouve {NbSim}.")
else:
    print(f"Il y a {NbSim} fichiers fort.q* (comme prévu).")


In [ ]:
# chargement des frames solution de Wave
frames = []
for i in range(NbSim+1):
      sol = solution.Solution(i, path=répertoire_résultat, file_format=format_fichier)
      frames.append(sol)

computation_domain, time_extent = check_output(frames)
dt  = time_extent['dt']
t_0 = time_extent['t_0']
t_f = time_extent['t_f']
dx_mnt, dy_mnt = computation_domain['dx'], computation_domain['dy']


In [ ]:
fgout_waves = fgout_tools.FGoutGrid(1, répertoire_résultat, format_fichier)

with contextlib.redirect_stdout(io.StringIO()):
        fgout_waves.read_fgout_grids_data()
        fg = fgout_waves.read_frame(1)
        
sol_0     = fg.B.T 
hauteur_0 = fg.h.T
eta_0     = sol_0+hauteur_0
surface_0 = len(hauteur_0[hauteur_0>0])*dx_mnt*dy_mnt
volume_0  = int(hauteur_0.sum()*dx_mnt*dy_mnt)
print("La profondeur du lac est {:.2f} m.".format(hauteur_0.max()))
print(f"Le volume du lac est {format_m(volume_0)} m³." )
print(f"La surface du lac est {format_m(surface_0)} m².")
zmin = eta_0.min()
zmax = eta_0.max()
print(f"Cote minimale du MNT : {zmin:.1f} m. Cote maximale du MNT : {zmax:.1f} m.")
print(f"Cote maximale : {lake['water_level']} m.")
Y_mnt = fg.Y
X_mnt = fg.X
pts_grille = np.column_stack([Y_mnt.ravel(), X_mnt.ravel()])
grid_shape  = X_mnt.shape

# Profil et animation 1D

In [ ]:
# définition du ndarray temps_in
temps_in = np.arange(t_0,t_f+dt,dt)
# grille
x_début = 965840
y_début = 6536175
x_fin   = 965730
y_fin   = 6536260

x_profil = np.linspace(x_début, x_fin, nb_grille)
y_profil = np.linspace(y_début, y_fin, nb_grille)
distance = ((x_profil-x_début)**2+(y_profil-y_début)**2)**0.5


In [ ]:
# définition des profils extraits du frame 0
ground_1d = gridtools.grid_output_2d(frames[0], fn_ground, x_profil, y_profil,
                                     levels='all', return_ma=True, method='linear')
eta_1d    = gridtools.grid_output_2d(frames[0], fn_eta,    x_profil, y_profil,
                                     levels='all', return_ma=True, method='linear')
z_coupe_min = ground_1d.min()-1
z_coupe_max = ground_1d.max()+0
print(f"solution au temps t = {frames[0].t} s")
fig = plt.figure(figsize=(10, 4))
axes = plt.subplot(1, 1, 1)
axes.set_ylim([z_coupe_min,z_coupe_max])
axes.fill_between(distance, ground_1d, z_coupe_min,
                  hatch='//', edgecolor='sienna', facecolor='white')
axes.plot(distance, ground_1d, color='sienna')
axes.fill_between(distance, eta_1d, ground_1d,
                  where=(eta_1d > ground_1d), color='skyblue')
axes.plot(distance, ma.masked_where(eta_1d == ground_1d, eta_1d), color='deepskyblue')



In [ ]:
# création des images
figures = Create_Animation(frames,x_profil,y_profil,config,temps_in)
animation_tools.interact_animate_figs(figures, manual=True)

In [ ]:
# Animation
images = animation_tools.make_images(figures)
anim   = animation_tools.animate_images(images, figsize=(12,4))
HTML(anim.to_jshtml())

In [ ]:
# export des images au format png
for i in range(len(figures)):
    figures[i].savefig(str(animation_dir / 'profil_vague')+str(i)+'.png', bbox_inches='tight',dpi=300)

In [ ]:
# export au format mp4
file_name = f'profil-vagues_{period_return}.mp4'
animation_tools.make_mp4( anim, file_name=video_dir /file_name)

# Énergie totale et flux dans le lac

Définition 
L'énergie de la vague d'impulsion est calculée comme la somme de l'énergie cinétique et l'énergie potentielle [J] :
$$
E_{lac}(t) = \frac{1}{2}\varrho g \int_{\mathcal S} \eta^2{\mathrm d} {\mathcal S} + \frac{1}{2}\varrho   \int_{\mathcal S} (h_0+\eta)v^2{\mathrm d} {\mathcal S}
$$
avec $  {\mathcal S}$ la surface du lac et $h_0$ la profondeur initiale du lac (avant impact), $v$ la vitesse de la vague et $\eta$ son amplitude.  L'énergie transmise par l'avalanche se calcule comme l'intégrale temporelle du flux d'énergie (puissance) [J] de l'avalanche aux frontières du domaine de l'avalanche ${\mathcal S}_a$ (de normale orientée $\boldsymbol{n}$) comprend une contribution cinétique :
$$
E_{cin,aval}(t) =   
\frac{1}{2}\varrho_a   \int_0^t\int_{{\mathcal S}_a}  \boldsymbol{u}^2 (\boldsymbol{u}\cdot\boldsymbol{n}) {\mathrm d}{\mathcal S}_a{\mathrm d} t,
$$
et une énergie potentielle :
$$
E_{pot, aval}(t) =  
\frac{1}{2}\varrho_a g \int_0^t\int_{{\mathcal S}_a}  (z+h-z_{lac})(\boldsymbol{u}\cdot\boldsymbol{n}){\mathrm d} {\mathcal S}_a{\mathrm d} t
$$


## Énergie avalanche

In [ ]:
reload_wave()
from module_waves import calculate_inflow_rate

In [ ]:
# lecture du fichier de masque du lac
contour_dict = create_plot_lake_contour_meshing(config_total,topo_file = None)
# calcul des flux entrant dans le lac via son contour
frame_temps, flux_énergie_cinétique, flux_énergie_potentielle, wave_flux_volume, wave_volume_lac = calculate_inflow_rate(contour_dict,config_total)

In [ ]:
# Analyse des puissances et énergies fournies par l'avalanche sur le contour du lac
ε = 5e-2 # seuil de détection du début et de la fin du flux de neige
# intégration des puissances
énergie_cinétique_fournie_avalanche   = np.array(list(accumulate(flux_énergie_cinétique, add)))*dt
énergie_potentielle_fournie_avalanche = np.array(list(accumulate(flux_énergie_potentielle, add)))*dt
# calcul de l'énergie totale
flux_énergie                = np.array(flux_énergie_cinétique) + np.array(flux_énergie_potentielle)
énergie_fournie_avalanche   = np.array(énergie_cinétique_fournie_avalanche)+np.array(énergie_potentielle_fournie_avalanche)
# détermination de la durée de l'avalanche
scaled_flux = flux_énergie_cinétique/flux_énergie_cinétique.max()

temps_arrivée_avalanche = frame_temps[scaled_flux> ε][0]
temps_fin_avalanche = frame_temps[scaled_flux> ε][-1]

fig, ((ax1, ax2)) = plt.subplots(1,2)
fig.suptitle(f"Puissance et énergie fournie par l'avalanche")
fig.set_figheight(5)
fig.set_figwidth(10)
ax1.plot(frame_temps,np.array(flux_énergie_cinétique)/1e6,label = r'$\dot E_c$')
ax1.plot(frame_temps,np.array(flux_énergie_potentielle)/1e6,label = r'$\dot E_p$')
ax1.plot(frame_temps,flux_énergie/1e6,'--k',label = r'$\dot E=\dot E_c+\dot E_p$')
ax1.set_ylabel(r"Puissance fournie au lac par l'avalanche [MW]")
ax1.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

ax2.plot(frame_temps,énergie_cinétique_fournie_avalanche/1e6,label = r'$ E_c$')
ax2.plot(frame_temps,énergie_potentielle_fournie_avalanche/1e6,label = r'$ E_p$')
ax2.set_ylabel(r"Énergie fournie par l'avalanche [MJ]")
ax2.plot(frame_temps,énergie_fournie_avalanche/1e6,'--k',label = r'$ E= E_c+ E_p$')
ax2.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2,label = r'flux avalancheux')

for ax in (ax1,ax2):
    ax.grid()
    ax.legend()
    ax.set_xlabel(r"$t$ [s]")
plt.tight_layout()
print(f"Flux avalancheux : t_début = {temps_arrivée_avalanche:.1f} s, t_fin = {temps_fin_avalanche:.1f} s. Le seuil de détection est {ε*100} %")
fig.savefig(images_dir / f"puissance-énergie-avalanche{period_return}.png", **figure_export_params)

In [ ]:
# On enregistre les temps d'arrivée et de fin de l'avalanche dans df_temps
# Ajout d'une ligne à df_temps avec écrasement au cas où il existe déjà une ligne
nouvelle_ligne = {'T': period_return, 'eps': ε,
                  't_déb': temps_arrivée_avalanche,
                  't_fin': temps_fin_avalanche}
mask_T = (df_temps['T'] == period_return)
if mask_T.any():
    df_temps.loc[mask_T, list(nouvelle_ligne.keys())] = list(nouvelle_ligne.values())
else:
    df_temps = pd.concat([df_temps, pd.DataFrame([nouvelle_ligne])], ignore_index=True)
# sauvegarde
df_temps.to_csv(df_temps_dir, index=False)
df_temps


## Énergie du lac

In [ ]:
# Masque du lac -> restreint les calculs à l'emprise du lac
# les valeurs sur tout le domaine de calcul sont indiquées par l'indice '_non_masquée'
# elles sont utiles pour les calculs et représentations graphiques hors du lac
Apply_mask = True

if Apply_mask:
    from scipy.interpolate import RegularGridInterpolator
    masque_file = raster_read_file(topo_dir / topo_files['mask_raster']) #'masque_qgis.asc')
    interp_masque = RegularGridInterpolator(
        (masque_file.y, masque_file.x), masque_file.Z, method='nearest',
        bounds_error=False, fill_value=0
    )
    masque_lac = ~interp_masque((Y_mnt, X_mnt)).astype(bool)
else:
    masque_lac = np.ones(X_mnt.shape, dtype=bool)


In [ ]:
# Calcul des énergies des vagues
énergie_dic, temps_wave = calculate_wave_energy(config,masque_lac.T)
énergie_tot_lac = énergie_dic['totale_lac']
énergie_cin_lac = énergie_dic['cinétique_lac']
énergie_pot_lac = énergie_dic['potentielle_lac']

In [ ]:
# Évolution de l'énergie totale des vagues
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_wave, énergie_tot_lac/1e6,label=r'$E=E_c+E_p$')
ax.plot(temps_wave, énergie_cin_lac/1e6,'--r',label=r'$E_c$')
ax.plot(temps_wave, énergie_pot_lac/1e6,'--k',label=r'$E_p$')
ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r'Énergie des vagues [MJ]')
ax.set_xlabel(r'Temps $t$ [s]')  
fig.legend(loc="upper center",ncol=3, frameon=False)

## Hauteur et volume dans le lac

In [ ]:
hauteur = énergie_dic['frames_lac']['hauteur']
variation_hauteur = np.array([hauteur[i].sum()*dx_mnt*dy_mnt  for i in range(NbSim+1)])

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(temps_wave, variation_hauteur,label=r'$E$')

ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_ylabel(r"volume d'eau $V$ du lac  [m$^3$]")
ax.set_xlabel(r'temps $t$ [s]')  
#fig.legend(loc="upper left",ncol=2)
jupyprint(f"L'augmentation de volume est de {format_m(variation_hauteur[-1]-variation_hauteur[1],0)}"+" $\\text{m}^3 $.")
jupyprint(f"L'accroissement maximal du volume est est {format_m(variation_hauteur.max()-variation_hauteur[1],0)}"+" $\\text{m}^3 $.")

## Efficience

In [ ]:
# Tracé
fig, ((ax1, ax2)) = plt.subplots(1,2)
#fig.suptitle(f"Énergie des vagues et énergie fournie par l'avalanche")
fig.set_figheight(5)
fig.set_figwidth(10)
ax1.plot(temps_wave,énergie_cin_lac/1e6,label = r'$E_C$')
ax1.plot(temps_wave,énergie_pot_lac/1e6,label = r'$E_p$')
ax1.plot(temps_wave,énergie_tot_lac/1e6,label = r'$E=E_c+E_p$')
ax1.set_ylabel(r"Énergies des vagues du lac [MJ]")
ax1.axvline(x=temps_arrivée_avalanche, ls=':',c='red',ymin=0, ymax=1,label = 'arrivée avalanche')
ax1.axvline(x=temps_fin_avalanche, ls=':',c='orange',ymin=0, ymax=1,label = 'arrêt avalanche')
ax1.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

ax2.plot(frame_temps,énergie_cinétique_fournie_avalanche/1e6,label = r'$ E_c$')
ax2.plot(frame_temps,énergie_potentielle_fournie_avalanche/1e6,label = r'$ E_p$')
ax2.plot(frame_temps,énergie_fournie_avalanche/1e6,label = r'$ E=E_c+E_p$')
ax2.set_ylabel(r"Énergies fournies par l'avalanche [MJ]")
ax2.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2)

for ax, texte in zip((ax1,ax2),['(a)','(b)']):
    ax.grid()
    ax.set_xlabel(r"$t$ [s]")
    ax.text(0.07, 0.91,  texte ,
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes,fontsize=14)
    ax.legend(loc='best')
plt.tight_layout()
# export
fig.savefig(images_dir / f"énergie-vague-avalanche_{period_return}.png", **figure_export_params)

In [ ]:
# Relation entre énergie des vagues et énergie fournie par l'avalanche
temps_interpolation = np.linspace(computation['t_0'],computation['t_max'],computation['nb_simul']+1)

interp_lac = np.interp(temps_interpolation,temps_wave,énergie_tot_lac)
interp_ava = np.interp(temps_interpolation,frame_temps,énergie_fournie_avalanche)

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(interp_ava/1e6,interp_lac/1e6,label=r'$E$')
ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"Énergie fournie   [MJ]")
ax.set_ylabel(r"Énergie totale du lac   [MJ]") 

In [ ]:
# Comparaison des efficiences énergétiques

efficience = np.divide(interp_lac, 
                       interp_ava, 
                       where=(interp_ava > 1e-3),
                       out=np.full_like(interp_lac, np.nan, dtype=float))
vague = énergie_dic['frames_lac']['vague']
vmax = np.nanmax([np.nanmax(vague[i])  for i in range(NbSim)] )
vmin = np.nanmin([np.nanmin(vague[i])  for i in range(NbSim)] )

# caractéristiques
print(f"énergie_tot_lac max = {énergie_tot_lac.max()/1e6:.3f} MJ")
print(f"énergie_pot_lac max = {énergie_pot_lac.max()/1e6:.3f} MJ")
print(f"énergie_cin_lac max = {énergie_cin_lac.max()/1e6:.3f} MJ")
print(f"amplitude des vagues : η_min ={vmax:.2f} m, η_min={vmin:.2f} m")


fig, ax = plt.subplots(figsize=(10,4))
mask_time = temps_interpolation >= temps_arrivée_avalanche
ax.plot(temps_interpolation[mask_time], efficience[mask_time],label=r'efficience $\zeta$')
ax.axvspan(temps_arrivée_avalanche, temps_fin_avalanche, ymin=0, ymax=1, color='grey',alpha=.2,label = r'phase avalancheuse $T=300$ ans')

ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$\zeta = E_{lac}/E_{aval.}$ ") 
ax.legend()
mask_time = (temps_interpolation>=temps_arrivée_avalanche) & (temps_interpolation<=temps_fin_avalanche)
ζ_max = np.nanmax(efficience[mask_time])
ζ_moy = efficience[mask_time].mean()
print(f"efficience maximale = {ζ_max:.3f}")
print(f"efficience moyenne  = {ζ_moy:.3f}")
fig.savefig(images_dir / f"efficience_énergie_{period_return}.png", **figure_export_params) 

In [ ]:
tab_efficience = pd.DataFrame(np.array([temps_interpolation,efficience]).T,columns=['t','efficience'])
tab_efficience.to_csv(export_dir / f"tableau_efficience_{period_return}.csv")
tab_efficience

In [ ]:
df_temps[df_temps['T']==100]['t_fin']

In [ ]:
df_temps[df_temps['T']==300]['t_déb'].iloc[0]

In [ ]:
data_300 = pd.read_csv(export_dir / f"tableau_efficience_300.csv")
data_100 = pd.read_csv(export_dir / f"tableau_efficience_100.csv")
 
 
fig, ax = plt.subplots(figsize=(10,4))
 
mask_time_300 = (data_300['t']>=df_temps[df_temps['T']==300]['t_déb'].iloc[0]) & (data_300['t']<=df_temps[df_temps['T']==300]['t_fin'].iloc[0])
mask_time_100 = (data_100['t']>=df_temps[df_temps['T']==100]['t_déb'].iloc[0]) & (data_100['t']<=df_temps[df_temps['T']==100]['t_fin'].iloc[0])
ax.plot(data_100['t'][mask_time_100 ],data_100['efficience'][mask_time_100 ],'--',label=r'$T=100$ ans')
ax.plot(data_300['t'][mask_time_300 ],data_300['efficience'][mask_time_300 ],label=r'$T=300$ ans')
ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$\zeta=E_{lac}/E_{aval.}$ ") 
ax.set_ylim([0,.5])

#ax.axvspan(df_temps[df_temps['T']==300]['t_déb'].iloc[0], df_temps[df_temps['T']==300]['t_fin'].iloc[0], ymin=0, ymax=1, color='grey',alpha=.2,label = r'phase avalancheuse $T=300$ ans')
ax.legend()
fig.savefig(images_dir / f"efficience_énergie_100-300.png", bbox_inches = 'tight',dpi=300)

# Jauges

In [ ]:
column_name =['level', 'time', 'h', 'hu', 'hv', 'eta' ]
fichier = 'gauge00001.txt'
data1 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)
fichier = 'gauge00002.txt'
data2 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)
fichier = 'gauge00003.txt'
data3 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)
fichier = 'gauge00004.txt'
data4 = pd.read_csv(répertoire_résultat / fichier, sep='\\s+',  comment='#', encoding='latin-1',names =  column_name, index_col=False)

étiquette = ['(1)','(2)','(3)','(4)']
 
def tracer_jauge(ax,data,k):
    ax.plot(data['time'], data['h'],label='CL')
    ax.set_xlabel(r"$t$ [s]")
    ax.set_ylabel(r"$h$ [m]")
    ax.grid()
    plt.text(0.9, 0.9, étiquette[k],
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes)
    
fig, ((ax1, ax2 ),(ax3, ax4 )) = plt.subplots(2,2)
fig.set_figheight(9)
fig.set_figwidth(10)
tracer_jauge(ax1,data1,0)
tracer_jauge(ax2,data2,1)
tracer_jauge(ax3,data3,2)
tracer_jauge(ax4,data4,3)
fig.savefig(images_dir / f'évolution_jauges_{period_return}.png',**figure_export_params)

In [ ]:
positions_jauges = []
if gauges['gauge_recording']:
        for k in range(len(gauges)-1):
            gaugeno = str(k)
            x = gauges[gaugeno]['x']
            y = gauges[gaugeno]['y']
            print(f"x = {x} y = {y}")
            positions_jauges.append([x,y])

positions_jauges

In [ ]:
# centre à partir duquel on trace les profils en long
xcenter, ycenter = 965800, 6536225-50
# Ensemble des données topo
topo_file = raster_read_file(topo_dir / lake['topography'])

fig, ax, x0, y0 = raster_plot_topo(topo_file, step=50)
ax.add_patch(plt.Rectangle((lake['xmin']-x0, lake['ymin']-y0),
                            width=lake['xmax']-lake['xmin'],
                            height=lake['ymax']-lake['ymin'],
                            ls="-", lw=2, ec="red", fc="none"))

# Zoom sur le lac avec une marge
margin = config['output']['carto_layout']['margin']  # mètres
text_m = 10
ax.set_xlim(lake['xmin'] - x0 - margin, lake['xmax'] - x0 + margin)
ax.set_ylim(lake['ymin'] - y0 - margin, lake['ymax'] - y0 + margin)
ax.plot([x_début-x0,x_fin-x0],[y_début-y0,y_fin-y0],'--k',linewidth = 0.8)

if gauges['gauge_recording']:
    for k in range(len(gauges)-1):
        x = gauges[str(k)]['x']
        y = gauges[str(k)]['y']
        ax.scatter(x-x0,y-y0,c = 'red',marker='o',s=12)
        ax.text(x-x0,y-y0+text_m,str(1+k),color = 'red')
ax.scatter(xcenter-x0,ycenter-y0,c = 'red',marker='+',s=15)
ax.text(xcenter-x0,ycenter-y0+text_m,'O',color = 'red')
fig.savefig(images_dir / "vue_domaine_calcul.png",**figure_export_params)
 


In [ ]:
# tracé des profils des 4 jauges
fig, _ = plot_four_gauge_profiles(config_total, xcenter, ycenter)

In [ ]:
fig.savefig(images_dir / 'profils_jauges.png',**figure_export_params)

# Hydrogramme des vagues sortant du domaine

## Débit de la lame passant par-dessus le remblai

In [ ]:
frame_temps, flux_volume = calculate_overflow_rate(config_total)

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
 
plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$Q$   [m$^3$/s]") 
ax.plot(frame_temps,flux_volume)
jupyprint(f"surverse = {np.trapz(flux_volume,frame_temps):.0f}  m $ ^3$ ")
print(f"zmax = {max([vague[i].max() for i in range(NbSim)] ):.2f} m")
print(f"zmin = {min([vague[i].min() for i in range(NbSim)] ):.2f} m")
print(f"Qmax = {np.max(flux_volume):0.1f} m3/s")

In [ ]:
fig.savefig(images_dir / f"débit_surverse_{period_return}.png",**figure_export_params)

In [ ]:
hydro = pd.DataFrame(np.array([frame_temps,flux_volume]).T,columns=['t','Q'])
hydro.to_csv(export_dir / f"hydrogramme_{period_return}.csv")
hydro

In [ ]:
data_300 = pd.read_csv(export_dir / f"hydrogramme_300.csv")
data_100 = pd.read_csv(export_dir / f"hydrogramme_100.csv")
 
 
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(data_100['t'],data_100['Q'],'--',label=r'$T=100$ ans')
ax.plot(data_300['t'],data_300['Q'],label=r'$T=300$ ans')
ax.grid(axis="y", color = 'gray', alpha = 0.25)
ax.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$Q$   [m$^3$/s]") 
ax.axvspan(df_temps[df_temps['T']==300]['t_déb'].iloc[0], df_temps[df_temps['T']==300]['t_fin'].iloc[0], ymin=0, ymax=1, color='grey',alpha=.2,label = r'phase avalancheuse $T=300$ ans')
ax.legend()
fig.savefig(images_dir / f"hydrogramme_100-300.png", **figure_export_params)

## Débit sortant d'une frontière du domaine de calcul

In [ ]:
temps_in = np.arange(t_0,t_f+dt,dt)
# grille : frontière nord 
x_début = lake['xmin']
y_début = lake['ymax']
x_fin   = lake['xmax']
y_fin   = lake['ymax']



x_profil = np.linspace(x_début, x_fin, nb_grille)
y_profil = np.linspace(y_début, y_fin, nb_grille)
distance = ((x_profil-x_début)**2+(y_profil-y_début)**2)**0.5
distance.shape

débit_sortant = []

for i in range(NbSim+1):
    qout = gridtools.grid_output_2d(frames[i], fn_hu, x_profil, y_profil, 
                                 levels='all',return_ma=True)
    débit_sortant.append(-1*qout.sum()*dx_mnt)

 

fig, ax = plt.subplots(figsize=(12,4))
 

plt.grid(axis="y", color = 'gray', alpha = 0.25)
plt.grid(axis="x", color = 'gray', alpha = 0.25)
ax.set_xlabel(r"$t$   [s]")
ax.set_ylabel(r"$Q$   [m$^3$/s]") 
ax.plot(temps_in,débit_sortant)

print(f"Qmax = {np.max(débit_sortant):0.1f} m3/s")
jupyprint(f"surverse = {int(np.array(débit_sortant).sum()*dt)}  m $ ^3$ ")

# Cartes et animation

In [ ]:
# Vue rapprochée du lac
#topo_lac = topotools.Topography(str(topo_dir / lake['topography']), topo_type=3)
topo_lac        = raster_read_file(topo_dir / lake['topography'])
filter_region   = (lake['xmin'], lake['xmax'], lake['ymin'], lake['ymax'])
topo_zoom       = topo_lac.crop(filter_region)
fig, ax, x0, y0 = raster_plot_topo(topo_zoom,contour_interval=5,step=100,figsize_width=5)

In [ ]:
# Définition des masques avec insertion de np.nan quand les hauteurs sont < epsilon
vague_masque_nan              = np.zeros((NbSim+1,)+hauteur_0.shape)
hauteur_masque_nan            = np.zeros((NbSim+1,)+hauteur_0.shape)
for i in range(NbSim+1):
      vague_masque_nan[i]   = énergie_dic['frames_domain']['vague'][i]
      hauteur_masque_nan[i] = énergie_dic['frames_domain']['hauteur'][i]
      hauteur_masque_nan[i][énergie_dic['frames_domain']['hauteur'][i]<computation['dry_limit']] = np.nan
      vague_masque_nan[i][énergie_dic['frames_domain']['hauteur'][i]<computation['dry_limit']]   = np.nan


In [ ]:
len(énergie_dic['frames_domain']['vague']) 

In [ ]:
len([i for i in range(NbSim+1)])

In [ ]:
énergie_dic['frames_domain']['vague'].shape

In [ ]:
# Animation des vagues
from matplotlib import animation
from IPython.display import HTML
matplotlib.rcParams['animation.embed_limit'] = 100
def make_wave_animation(frames, vague_masque, hauteur_masque, topo_zoom, lake, x0, y0,
                        step = 20, contour_interval=5,
                        vmax=0.5, fps=5, figsize_width = 10):
    """
    Animation vue de dessus de la vague (eta - eta_0).
    
    Paramètres
    ----------
    frames       : liste des Solution clawpack
    vague_masque : array (NbSim, ny, nx) pré-calculé (valeurs NaN hors eau)
    topo_zoom    : objet retourné par raster_read_file (topo recadrée sur le lac)
    lake         : dict avec xmin, xmax, ymin, ymax, water_level
    x0, y0       : coin SW de topo_zoom (retournés par raster_plot_topo)
    vmax         : amplitude max de la colorbar (symétrique ±vmax)
    fps          : images par seconde
    figsize      : taille de la figure
    """
    xmin, xmax = lake['xmin'], lake['xmax']
    ymin, ymax = lake['ymin'], lake['ymax']
    vmax = np.nanmax([np.nanmax(vague_masque[i])  for i in range(NbSim)] )
    vmin = np.nanmin([np.nanmin(vague_masque[i])  for i in range(NbSim)] )

    # --- figure de base (fond topo) ---
    fig, ax, x0, y0 = raster_plot_topo(topo_zoom, contour_interval= contour_interval, step=step,
                                 figsize_width=figsize_width)
    ax.set_xlim(xmin - x0  , xmax - x0  )
    ax.set_ylim(ymin - y0  , ymax - y0  )

    # --- artiste imshow (initialisé sur la première frame) ---
    cmap_lac = ListedColormap([(0, 0, 0, 0),(0.5, 0.7, 1.0, 0.8)])
    im = ax.imshow(hauteur_masque[0], origin='lower',
                   extent=[xmin - x0, xmax - x0, ymin - y0, ymax - y0],
                   cmap=cmap_lac, vmin=0, vmax=1, zorder=2) #,


    im = ax.imshow(vague_masque[0], origin='lower',
               extent=[xmin-x0, xmax-x0, ymin-y0, ymax-y0],
               cmap='coolwarm', vmin=-vmax, vmax=vmax, alpha=0.85, zorder=3)



    fig.colorbar(im, ax=ax, shrink=0.6, label=r'$\eta - \eta_0$ (m)')

    title_text = ax.set_title(rf"$t = {frames[0].t:.1f}$ s")

    # --- fonction de mise à jour ---
    def update(i):
        im.set_data(vague_masque[i])
        # par-dessus : vague animée
        
        title_text.set_text(rf"$t = {frames[i].t:.1f}$ s")
        return im, title_text

    anim = animation.FuncAnimation(fig, update, frames=len(frames),
                                   interval=1000 // fps, blit=True)
    plt.close(fig)   # évite l'affichage statique en plus
    return anim

anim = make_wave_animation(frames, vague_masque_nan, hauteur_masque_nan, topo_zoom, lake, x0, y0 , figsize_width = 7,step=100)
HTML(anim.to_jshtml())


In [ ]:
animation_tools.make_mp4(anim, file_name = video_dir / f"vagues_{config['scenario']['period_return']}.mp4")


In [ ]:
vmax = max([vague[i].max() for i in range(NbSim)] )
vmin = min([vague[i].min() for i in range(NbSim)] )
print(f"ampleur maximale des vagues {vmax:.1f} m")
print(f"creux maximal des vagues {vmin:.1f} m")

# Boucle itérative

In [ ]:
# Boucle de simulations
rendement_max_list    = []
rendement_moy_list    = []
variation_volume_list = []
Qmax_list             = []
amplitude_max_list    = []
amplitude_min_list    = []
volume_surverse_list  = []

damping_list = np.linspace(0.1, 0.4, 25)
#damping_list = [0.3]
# Points de la grille de post-traitement (calculés une seule fois hors boucle)
# pts_grille = np.column_stack([Y_grille.ravel(), X_grille.ravel()])
# grid_shape = X_grille.shape

for damp in tqdm(damping_list):
    config['computation']['damping'] = float(damp)
    config_total['waves']['computation']['damping'] = float(damp)
    print('-----------------------------------------')
    print(f"amortissement = {config['computation']['damping']}")
    # sauvegarde la configuration
    with open(configuration_file, 'w') as f:
        safe_dump(config, f)
    # exécution
    make_output(config)

    # post traitement
    NbSim = computation['nb_simul'] 
    # Calcul des énergies des vagues
    énergie_dic, temps_wave = calculate_wave_energy(config,masque_lac.T)
    énergie_tot_lac = énergie_dic['totale_lac']
    énergie_cin_lac = énergie_dic['cinétique_lac']
    énergie_pot_lac = énergie_dic['potentielle_lac']
    hauteur         = énergie_dic['frames_lac']['hauteur']

    # Accroissement du volume dans le lac
    variation_volume = np.array([hauteur[i].sum()*dx_mnt*dy_mnt for i in range(NbSim+1)]) - volume_0
    ΔV_max           = variation_volume.max()
    variation_volume_list.append(ΔV_max)
    jupyprint(f"L'accroissement maximal du volume est {format_m(ΔV_max, 0)}"+" $\\text{m}^3 $.")

    # Calcul de l'efficience
    interp_lac = np.interp(temps_interpolation, temps_wave, énergie_tot_lac)
    #interp_lac_variante = np.interp(temps_interpolation, temps_in, énergie_tot_lac)
    efficience = np.divide(interp_lac, interp_ava,
                           where=(interp_ava > 1e-3),
                           out=np.full_like(interp_lac, np.nan, dtype=float))
    print(f"énergie_tot_lac max = {énergie_tot_lac.max()/1e6:.3f} MJ")
    # À ajouter dans les deux contextes
    print(f"énergie_pot_lac max = {énergie_pot_lac.max()/1e6:.3f} MJ")
    print(f"énergie_cin_lac max = {énergie_cin_lac.max()/1e6:.3f} MJ")
   
    mask_time = (temps_interpolation>=temps_arrivée_avalanche) & (temps_interpolation<=temps_fin_avalanche)
    ζ_max = np.nanmax(efficience[mask_time])
    ζ_moy = efficience[mask_time].mean()
    rendement_moy_list.append(ζ_moy)
    rendement_max_list.append(ζ_max)
    vague = énergie_dic['frames_lac']['vague']
    η_max = max([vague[i].max() for i in range(NbSim)])
    η_min = min([vague[i].min() for i in range(NbSim)])

    # Débit et volume de surverse
    frame_temps, flux_volume = calculate_overflow_rate(config_total)
    volume_surverse = int(np.array(flux_volume).sum() * dt)
    Qp_surverse     = np.max(flux_volume)
    print(f"volume de surverse = {volume_surverse:.0f}  m³")
    print(f"débit de surverse Qmax = {Qp_surverse:0.1f} m³/s")
    print(f"amplitude des vagues : ηmax = {η_max:.3f} m")
    print(f"                       ηmin = {η_min:.3f} m")
    print(f"efficience max : ζ_max = {ζ_max:.3f}")
    print(f"efficience moy : ζ_moy = {ζ_moy:.3f}")
    amplitude_max_list.append(η_max)
    amplitude_min_list.append(η_min)
    volume_surverse_list.append(volume_surverse)
    Qmax_list.append(Qp_surverse)

In [ ]:
résultats_boucle = pd.DataFrame(np.array([damping_list,rendement_max_list,rendement_moy_list,
variation_volume_list,
Qmax_list,
amplitude_max_list,
amplitude_min_list,
volume_surverse_list]).T,columns=['amortissement','efficience_max','efficience_moy','variation_volume','Qmax','eta_max','eta_min','volume_surverse'])
résultats_boucle

In [ ]:
export_dir    = proj_dir / 'Export'
résultats_boucle.to_csv(export_dir / f'résultats_incertitude_{period_return}.csv')

In [ ]:
résultats_boucle[résultats_boucle['amortissement']==0.2]

In [ ]:
résultats_boucle[résultats_boucle['amortissement']==0.25]

In [ ]:
fig,((ax1, ax2),(ax3, ax4))  = plt.subplots(2, 2, figsize=(10,10))

ax1.scatter(résultats_boucle['amortissement']*1000, résultats_boucle['volume_surverse'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95) #,label = 'hydro')
ax1.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax1.set_ylabel(r'Volume de surverse $V$ [m$^3$]' )
ax1.grid(True, alpha=0.3,  which="both", ls="-")

ax2.scatter(résultats_boucle['amortissement']*1000, résultats_boucle['eta_max'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95) #,label = 'hydro')
ax2.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax2.set_ylabel(r'Amplitude des vague $A$ [m]' )
ax2.grid(True, alpha=0.3,  which="both", ls="-")

ax3.scatter(résultats_boucle['amortissement']*1000, résultats_boucle['efficience_moy'], 
            marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95)  
# ax3.scatter(résultats_boucle['amortissement']*1000, résultats_boucle['efficience_max'], 
#             marker = "s",edgecolors='blue', s=45, color='skyblue', alpha = 0.95) 
ax3.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax3.set_ylabel(r"Transfert d'énergie $\zeta$  " )
ax3.grid(True, alpha=0.3,  which="both", ls="-")

ax4.scatter(résultats_boucle['amortissement']*1000, résultats_boucle['Qmax'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95) #,label = 'hydro')
ax4.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax4.set_ylabel(r'Débit maximal de surverse $Q_{max}$ [m$^3\cdot$s$^{-1}$]' )
ax4.grid(True, alpha=0.3,  which="both", ls="-")



In [ ]:
fig.savefig(images_dir / f"résultat_incertitudes_{period_return}.png",dpi=300,bbox_inches = 'tight')

In [ ]:
data_300 = pd.read_csv(export_dir / f"résultats_incertitude_300.csv")
data_100 = pd.read_csv(export_dir / f"résultats_incertitude_100.csv")
 
fig,((ax1, ax2),(ax3, ax4))  = plt.subplots(2, 2, figsize=(8,8))
ax1.scatter(data_300['amortissement']*1000, data_300['volume_surverse'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95,label = r'$T=300$ ans')
ax1.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax1.set_ylabel(r'Volume de surverse $V$ [m$^3$]' )
ax1.grid(True, alpha=0.3,  which="both", ls="-")

ax2.scatter(data_300['amortissement']*1000, data_300['eta_max'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95)  
ax2.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax2.set_ylabel(r'Amplitude des vague $\eta$ [m]' )
ax2.grid(True, alpha=0.3,  which="both", ls="-")

#ax3.scatter(data_300['amortissement']*1000, data_300['efficience_max'], marker = "+",edgecolors='blue', s=45, color='skyblue', alpha = 0.95,label=r"$\zeta_{pic}$") 
ax3.scatter(data_300['amortissement']*1000, data_300['efficience_moy'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95,label=r"$\zeta_{moy}$")  
ax3.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax3.set_ylabel(r"Transfert d'énergies $\zeta$  " )
ax3.grid(True, alpha=0.3,  which="both", ls="-")

ax4.scatter(data_300['amortissement']*1000, data_300['Qmax'], marker = "o",edgecolors='blue', s=45, color='skyblue', alpha = 0.95) #,label = 'hydro')
ax4.set_xlabel(r'$\varrho_a$ [kg$\cdot$m$^{-3}$]') 
ax4.set_ylabel(r'Débit maximal de surverse $Q_{max}$ [m$^3\cdot$s$^{-1}$]' )
ax4.grid(True, alpha=0.3,  which="both", ls="-")

# T = 100
ax1.scatter(data_100['amortissement']*1000, data_100['volume_surverse'], marker = "o",edgecolors='red', s=45, color='orange', alpha = 0.95,label = r'$T=100$ ans')
ax1.legend()

ax2.scatter(data_100['amortissement']*1000, data_100['eta_max'], marker = "o",edgecolors='red', s=45, color='orange', alpha = 0.95)  

#ax3.scatter(data_100['amortissement']*1000, data_100['efficience_max'], marker = "+",edgecolors='red', s=45, color='orange', alpha = 0.95,label=r"$\zeta_{pic}$")
ax3.scatter(data_100['amortissement']*1000, data_100['efficience_moy'], marker = "o",edgecolors='red', s=45, color='orange', alpha = 0.95,label=r"$\zeta_{moy}$")  
 
#ax3.legend()

ax4.scatter(data_100['amortissement']*1000, data_100['Qmax'], marker = "o",edgecolors='red', s=45, color='orange', alpha = 0.95) 

ax4.grid(True, alpha=0.3,  which="both", ls="-")

plt.tight_layout()

fig.savefig(images_dir / f"résultat_incertitudes_100-300.png", bbox_inches = 'tight',dpi=300)